# Lesson 12 — Reaction-Rate Tallies, $\bar{\eta}$, and $k_{\infty}$

**Heterogeneous variant:** a centered sphere of natural uranium metal surrounded
by water, with the original homogeneous model retained for comparison.

Building on the previous lessons, we will

1. Add explicit `nu-fission` and `absorption` tallies and divide by flux to obtain
   spectrum-weighted $\overline{\nu\Sigma_f}$ and $\overline{\Sigma_a}$.
2. Define the production/absorption ratio with its **absorption denominator stated**.
3. Sweep the H/U **atom** ratio and compare the heterogeneous and original
   homogeneous models at the same H/U.
4. Turn the external-source calculation into a $k$-eigenvalue calculation and
   see why the two spectra and the two ratios are not automatically identical.

**Draft for instructor review.** Select a Python kernel containing OpenMC,
NumPy, pandas, and Matplotlib. The OpenMC executable and a continuous-energy
neutron-data library, including `c_H_in_H2O`, must be available. Activate your
OpenMC environment before launching Jupyter; its `OPENMC_CROSS_SECTIONS`
variable should name `cross_sections.xml`.

This notebook is standalone: it does not execute or modify another lesson.
The comparison sweep writes XML/HDF5 files to separate case directories under
the scratch directory printed below.

The heterogeneous model uses nominal densities of **19.1 g/cm³ for uranium
metal** and **1.0 g/cm³ for water**, with no thermal expansion model. Uranium
uses OpenMC's natural isotopic abundances. Water retains the original H1/O16
composition, and oxygen belongs to water, not added UO₂. Setting
`heterogeneous=False` recovers the original **1 g/cm³ homogeneous mixture**.

The two models have the same H/U and outer radius, but **not the same total
atom inventories or volume-averaged density**. With uranium volume fraction
$f_U$, an inventory-matched homogeneous mixture would instead have density
$f_U\rho_U+(1-f_U)\rho_w$; that is not the original homogeneous baseline used here.

Density reference: [Royal Society of Chemistry: uranium](https://periodic-table.rsc.org/element/92/uranium).

In [ ]:
import openmc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# GLOBAL CONSTANTS
TEMPERATURE = 294.0
RADIUS = 1.0  # cm, unchanged outer reflective sphere
VOLUME = 4.0 * np.pi * RADIUS**3 / 3.0
URANIUM_DENSITY = 19.1  # g/cm3, nominal natural uranium metal
WATER_DENSITY = 1.0     # g/cm3
K_B = 8.617333262e-5  # eV/K
ENERGY_EDGES = np.geomspace(1e-5, 2e7, 1001)  # geomspace like logspace but with absolute points
ENERGY = np.sqrt(ENERGY_EDGES[:-1] * ENERGY_EDGES[1:])  # "center energy"
DELTA_E = np.diff(ENERGY_EDGES)
DELTA_U = np.log(ENERGY_EDGES[1:] / ENERGY_EDGES[:-1])

PURPLE = "#512888"
ORANGE = "#CA7C1B"

RUN_DIR = Path("lesson_12_sphere_runs").resolve()
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Scratch directory: {RUN_DIR}")

## 1. Tally reaction rates first; derive effective cross sections second

For the spectrum integrated over the **entire outer sphere**, define

$$\Phi=\int_V\!\int_0^\infty\phi(\mathbf r,E)\,dE\,dV,\qquad
 P=\int_V\!\int_0^\infty\nu\Sigma_f(\mathbf r,E)\phi(\mathbf r,E)\,dE\,dV,$$
$$A=\int_V\!\int_0^\infty\Sigma_{a}(\mathbf r,E)\phi(\mathbf r,E)\,dE\,dV.$$

Then

$$
\begin{split}
\overline{\nu\Sigma_f}         &= P/\Phi \\
\overline{\Sigma_{a}}          &= A/\Phi \\
\overline{\eta}               &= P/A \, .
\end{split}
$$

OpenMC `nu-fission` and `absorption` are **reaction-rate scores**, not direct
cross sections. Dividing either by the corresponding `flux` tally yields an
effective macroscopic cross section in cm$^{-1}$.

Here $A=A_U+A_{\mathrm{water}}$, not uranium absorption alone. Tallies have
**no spatial filter**, so both regions contribute to the same tally bin.
The spectrum is divided by the full outer-sphere volume, as before.
This also retains OpenMC's uncertainty estimate for the whole-sphere spectrum
without summing separate cell standard deviations.

Tally reference: [OpenMC: specifying tallies](https://docs.openmc.org/en/stable/usersguide/tallies.html).

## 2. Set the uranium radius from H/U

Let $n_H^{w}$ be the hydrogen atom density in water and $n_U^{m}$ the total
uranium atom density in metal. For outer radius $R$ and uranium radius $r_U$,
the **whole-sphere atom ratio** is

$$
\frac{H}{U}
=\frac{n_H^{w}(V-V_U)}{n_U^{m}V_U}
=\frac{n_H^{w}(R^3-r_U^3)}{n_U^{m}r_U^3}.
$$

Therefore,

$$
f_U\equiv\frac{V_U}{V}
=\frac{n_H^{w}}{n_H^{w}+(H/U)n_U^{m}},
\qquad
\boxed{r_U=R\,f_U^{1/3}}.
$$

The code obtains both atom densities from the actual OpenMC materials, in the
same units (atom/b-cm). Water occupies only $r_U<r<R$; its displaced volume
is not counted in H/U. `H_to_U=None` retains the water-only case.

The outer boundary remains reflective; the uranium–water interface is
transmitting. The original **coded source** is preserved: an isotropic
**2 MeV point source at the origin**, now inside the uranium sphere.
(The supplied notebook's prose said 1 MeV, but its code used 2 MeV.)

As before, `multiplying=True` transports fission descendants, sampled from
fission data rather than forced to the external-source energy. The comparison
sweep retains this setting. **Check subcriticality before running a multiplying
fixed-source case**, especially after changing radius, composition, or density;
an eigenvalue check is not included in this notebook. Use `multiplying=False`
to suppress fission descendants. A fixed-source $P/A$ is not automatically
$k_{\infty}$.

Atom-density reference: [OpenMC: Material.get_nuclide_atom_densities](https://docs.openmc.org/en/stable/pythonapi/generated/openmc.Material.html#openmc.Material.get_nuclide_atom_densities).

In [ ]:
def make_model(H_to_U=100.0, temperature=294, bound_hydrogen=True,
               multiplying=False, particles=2000, batches=40, seed=11001,
               heterogeneous=True):
    """Return (model, materials, cells); the last two entries are lists.

    H_to_U is a positive, finite whole-sphere atom ratio, or None for water.
    heterogeneous=False retains the original 1 g/cm3 homogeneous mixture.
    """
    if H_to_U is not None:
        H_to_U = float(H_to_U)
        if not np.isfinite(H_to_U) or H_to_U <= 0.0:
            raise ValueError("H_to_U must be positive and finite, or None for water.")

    sphere = openmc.Sphere(r=RADIUS, boundary_type="reflective")

    if H_to_U is None or heterogeneous:
        water = openmc.Material(name="water")
        water.add_nuclide("H1", 2.0)
        water.add_nuclide("O16", 1.0)
        water.set_density("g/cm3", WATER_DENSITY)
        water.temperature = float(temperature)
        if bound_hydrogen:
            water.add_s_alpha_beta("c_H_in_H2O")

        if H_to_U is None:
            water.volume = VOLUME
            materials = [water]
            cells = [openmc.Cell(name="water", fill=water, region=-sphere)]
        else:
            uranium = openmc.Material(name="natural U metal")
            uranium.add_element("U", 1.0)  # OpenMC's natural isotopic abundances
            uranium.set_density("g/cm3", URANIUM_DENSITY)
            uranium.temperature = float(temperature)

            n_H = water.get_nuclide_atom_densities()["H1"]
            n_U = sum(uranium.get_nuclide_atom_densities().values())
            f_U = n_H / (n_H + H_to_U * n_U)
            r_U = RADIUS * np.cbrt(f_U)
            uranium.volume = VOLUME * f_U
            water.volume = VOLUME - uranium.volume

            fuel_sphere = openmc.Sphere(r=float(r_U))  # transmission interface
            fuel_cell = openmc.Cell(name="uranium sphere", fill=uranium,
                                    region=-fuel_sphere)
            water_cell = openmc.Cell(name="water shell", fill=water,
                                     region=+fuel_sphere & -sphere)
            materials = [uranium, water]
            cells = [fuel_cell, water_cell]
    else:
        material = openmc.Material(name="natural U + water")
        material.add_nuclide("H1", H_to_U)
        material.add_nuclide("O16", H_to_U / 2.0)
        material.add_element("U", 1.0)
        material.set_density("g/cm3", 1.0)  # original homogeneous density
        material.temperature = float(temperature)
        material.volume = VOLUME
        if bound_hydrogen:
            material.add_s_alpha_beta("c_H_in_H2O")
        materials = [material]
        cells = [openmc.Cell(name="homogeneous mixture", fill=material,
                             region=-sphere)]

    settings = openmc.Settings()
    settings.run_mode = "fixed source"
    settings.source = openmc.IndependentSource(
        space=openmc.stats.Point((0.0, 0.0, 0.0)),
        energy=openmc.stats.Discrete([2e6], [1.0]),
    )
    settings.particles = int(particles)
    settings.batches = int(batches)
    settings.seed = int(seed)
    settings.create_fission_neutrons = bool(multiplying)

    model = openmc.Model(geometry=openmc.Geometry(cells),
                         materials=openmc.Materials(materials), settings=settings)

    # No spatial filter: score the whole sphere, including both materials.
    spectrum = openmc.Tally(name="spectrum")
    spectrum.filters = [openmc.EnergyFilter(ENERGY_EDGES)]
    spectrum.scores = ["flux"]
    spectrum.estimator = "tracklength"

    integral = openmc.Tally(name="integral reaction rates")
    integral.scores = ["flux", "nu-fission", "absorption"]
    integral.estimator = "tracklength"

    model.tallies = openmc.Tallies([spectrum, integral])
    return model, materials, cells

In [ ]:
def get_results(sp_path):
    with openmc.StatePoint(sp_path) as sp:
        spectrum = sp.get_tally(name="spectrum")
        mean = spectrum.get_values(scores=["flux"], value="mean").ravel().copy()
        sd = spectrum.get_values(scores=["flux"], value="std_dev").ravel().copy()
        results = {"bin_flux": mean, "bin_sd": sd,
                   "phi_E": mean / (VOLUME * DELTA_E),
                   "phi_u": mean / (VOLUME * DELTA_U)}
        
        integral = sp.get_tally(name="integral reaction rates")
        means = {score: float(integral.get_values(scores=[score], value="mean").sum())
                     for score in integral.scores}
        Phi, P, A = (means[score] for score in ["flux", "nu-fission", "absorption"])
        
        results["Phi"] = means["flux"]
        results["P"] = means["nu-fission"]
        results["A"] = means["absorption"]
        
        return results

## 3. Exploring $\bar{\eta}$ with H/U

Run both geometries at each original H/U value, retaining the same source,
particle count, batch count, and fission-descendant setting. The table reports
$P/(A_U+A_{\mathrm{water}})$ and the uranium radius used in the heterogeneous
case. Results are also retained in `case_results[(H_to_U, geometry)]` for
subsequent spectral comparisons.

Set `GEOMETRIES = ("heterogeneous",)` to run only the new model.

In [ ]:
H_TO_U_VALUES = [100, 50, 25, 20, 15, 10, 5, 4, 3]
GEOMETRIES = ("homogeneous", "heterogeneous")
MULTIPLYING = True  # original sweep setting; requires subcritical cases

rows = []
case_results = {}
for HU in H_TO_U_VALUES:
    row = {"H/U": HU}
    for geometry in GEOMETRIES:
        model, materials, cells = make_model(
            H_to_U=HU, bound_hydrogen=True, multiplying=MULTIPLYING,
            heterogeneous=(geometry == "heterogeneous"),
        )
        if geometry == "heterogeneous":
            row["U radius [cm]"] = np.cbrt(3.0 * materials[0].volume / (4.0 * np.pi))

        case_dir = RUN_DIR / f"{geometry}_HU_{HU:g}"
        case_dir.mkdir(parents=True, exist_ok=True)
        sp_path = model.run(cwd=case_dir, output=False)
        results = get_results(sp_path)
        case_results[(HU, geometry)] = results
        row[f"eta_{geometry}"] = results["P"] / results["A"]
        print(f"H/U={HU:g}  {geometry:13s}  eta={row[f'eta_{geometry}']:.6f}")
    rows.append(row)

comparison = pd.DataFrame(rows).set_index("H/U")
comparison